# REBULEX — reranker ölçümü (Colab)

`rerank.py` embedding aramasının getirdiği ilk 100 kararı bir reranker'la yeniden sıralar ve ölçer.
İndeks yeniden kurulmaz; yalnızca sorgu vektörleri hesaplanır. Kod GitHub'dan (`metin-temizleme` dalı),
veriler Drive'daki tek bir zip'ten gelir.

**Başlamadan önce**
1. `Çalışma zamanı > Çalışma zamanı türünü değiştir` → **GPU (A100 ya da L4)**.
2. Drive'da `REBULEX/rerank_veri.zip` olmalı (bilgisayardaki `data/rerank_veri.zip`).
3. EmbeddingGemma adayları için soldaki 🔑 *Secrets* bölümünde `HF_TOKEN` olmalı (notebook erişimi açık).

**Akış:** 1–4. hücreler bir kez. Sonra 5 (duman testi), 6 (pilot), 7 (Drive'a kaydet), 8 (sonuçlar).

## 1. Drive'ı bağla ve GPU'yu kontrol et

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Ayarlar
`REBULEX_RERANK_BATCH`: aynı anda puanlanan çift sayısı. A100/L4 için 32 uygundur; bellek hatası olursa 16'ya düşür.

In [ ]:
import os

DRIVE_DIR = "/content/drive/MyDrive/REBULEX"
LOCAL_DATA = "/content/data"
REPO = "https://github.com/ahmetege0/rebulex.git"
BRANCH = "metin-temizleme"

os.environ["REBULEX_DATA"] = LOCAL_DATA     # config.py veriyi buradan okur
os.environ["REBULEX_RERANK_BATCH"] = "32"
os.environ["REBULEX_DTYPE"] = "bfloat16"    # sorgu vektörlerini çıkaran embedding modelleri için

## 3. Kodu GitHub'dan çek ve kur
Kütüphane sürümleri bilgisayardakiyle aynı tutulur (sentence-transformers 6.0.1, transformers 5.12.1).

In [ ]:
if not os.path.exists("/content/rebulex"):
    !git clone -q -b {BRANCH} {REPO} /content/rebulex
else:
    !cd /content/rebulex && git fetch -q && git reset -q --hard origin/{BRANCH}
%cd /content/rebulex
!git log --oneline -1
!pip install -q -r requirements.txt sentence-transformers==6.0.1 transformers==5.12.1
import sentence_transformers, transformers, torch
print("sentence-transformers", sentence_transformers.__version__, "| transformers", transformers.__version__,
      "| torch", torch.__version__, "| GPU:", torch.cuda.is_available())

from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN yüklendi")
except Exception as e:
    print("HF_TOKEN yok; EmbeddingGemma adayları çalışmaz:", type(e).__name__)

## 4. Verileri Drive'dan Colab diskine aç
Zip'te yalnızca gerekenler var: 2000'lik havuz, test setleri, `experiments.csv` ve üç aday kaynağının
`chunks` / `embeddings` / `index_info` dosyaları.

In [ ]:
if not os.path.exists(f"{LOCAL_DATA}/experiments.csv"):   # oturum içinde eklenen satırlar ezilmesin
    !unzip -q -o {DRIVE_DIR}/rerank_veri.zip -d {LOCAL_DATA}
!find {LOCAL_DATA} -type f | sort

## 5. Duman testi (birkaç dakika)
Cosmos 64'lük parçalarla, yalnızca v2: önce `yok` (reranker yok, embedding sırası), sonra ModernBERT-TR.
`yok` satırındaki `ilk5_genis_%` gece koşusundaki değerle (64.4) aynı çıkmalı.

In [ ]:
!python -u rerank.py --kaynak temiz/cosmos_e5 --reranker yok modernbert_tr --test-set v2

## 6. Pilot: üç aday kaynağı, iki test seti, ModernBERT-TR
Duman testinde bitenler atlanır. İşe yararsa diğer rerankerlar `--reranker bge_m3 qwen3_06b` ile eklenir.

In [ ]:
!python -u rerank.py --reranker yok modernbert_tr

## 7. Sonuçları Drive'a kaydet
`rerank` klasörü ve `experiments.csv`, Drive'da `REBULEX/rerank_sonuc/` altına kopyalanır.
Drive'daki eski `data/experiments.csv`'ye dokunulmaz.

In [ ]:
import shutil

out = f"{DRIVE_DIR}/rerank_sonuc"
shutil.copytree(f"{LOCAL_DATA}/n2000/rerank", f"{out}/rerank", dirs_exist_ok=True)
shutil.copy(f"{LOCAL_DATA}/experiments.csv", f"{out}/experiments.csv")
!find {out} -type f | sort

## 8. Sonuçlar (geniş ölçü)

In [ ]:
import pandas as pd

exp = pd.read_csv(f"{LOCAL_DATA}/experiments.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
rows = exp[exp["not"].str.startswith("rerank")]
cols = ["not", "deneme", "test_seti", "reranker", "ilk1_genis_%", "ilk5_genis_%", "ilk10_genis_%",
        "ilk20_genis_%", "ilk100_genis_%", "mrr_genis", "rerank_sn"]
rows[cols]